In [9]:
"""
This script identifies movebank studies to use in the project.

It queries all the studies in movebank that have download access and
GPS, downloads a sample of tracks, and computes the median interval between
points.

Saves the studies to data/movebank/studies/studies_to_use.csv
"""

import os
import pandas as pd
import numpy as np
from dataclasses import asdict
import requests
from io import StringIO
import seaborn as sns
import hashlib

pd.set_option('display.max_columns', None)

import util as bw_utils

from scales_human_mobility.utils import scale_by_scale_optim
from scales_human_mobility.utils import compute_intervals
from scales_human_mobility.utils import scales_of_mobility
from scales_human_mobility.utils import scale_fitter_no_grid
from scales_human_mobility.utils import utils

pd.set_option('display.max_columns', None)


BASE_URL = "https://www.movebank.org/movebank/service/direct-read"
AUTH = ('bxwallace', 'House41!?')
FOLDER = "../data/movebank/studies/"


In [3]:
"""
Downloads descriptions all the studies from movebank.
"""

params = {
    "entity_type": "study",
    "i_have_download_access": "true",
}
# We have to do this whole stupid double-request to accept the response.
# We request the data once here, then hash the response text, which is about
# accepting use terms, and then we send it back.
response = requests.get(
    BASE_URL,
    params=params,
    auth=AUTH,
)
# I'm not totally sure what this line does.
response.raise_for_status()

all_studies = pd.read_csv(
    StringIO(response.text),
    parse_dates=['timestamp_last_deployed_location', 'timestamp_first_deployed_location'],
    skiprows=[466], #this one has a bad timestamp that breaks the data
)

all_studies.to_cvs(FOLDER + "all_studies_unfiltered.csv")

In [174]:
# """
# Filter the studies
# """
# MIN_POINTS = 10000

# studies = all_studies[
#     # only keep studies with GPS
#     all_studies["sensor_type_ids"].str.contains("GPS", na=False) &
#     # at least 3 individuals
#     (all_studies["number_of_individuals"] >= 1) &
#     # at least 10,000 events
#     (all_studies["number_of_deployed_locations"] >= MIN_POINTS) &
#     # at least 1 year of data
#     (all_studies['timestamp_last_deployed_location'] - all_studies['timestamp_first_deployed_location'] >= pd.Timedelta(days=365))
# ].reset_index(drop=True).copy()

# len(studies)

320

In [175]:
def get_events(study_id, full=False):
    """
    Queries movebank for all the study data for `study_id`.

    Expects the following to be defined in the environment.

    BASE_URL = "https://www.movebank.org/movebank/service/direct-read"
    AUTH = ([username], [password]) # movebank credentials.

    You should set these in your notebook and not here.
    """
    params = {
        "entity_type": "event",
        "study_id": study_id,
        # "attributes": "all",
    }
    if not full:
        # Prebuilt event filter to get only the last 30 days of events.
        params["event_reduction_profile"] = "EURING_03"

    else:
        params["attributes"] = "all"
    
    # We have to do this whole stupid double-request to accept the response.
    # We request the data once here, then hash the response text, which is about
    # accepting use terms, and then we send it back.
    response = requests.get(
        BASE_URL,
        params=params,
        auth=AUTH,
    )
    # I'm not totally sure what this line does.
    response.raise_for_status()

    # Make a hash of the response text to show that we read it and accept the terms.
    params["license-md5"] = hashlib.md5(response.text.encode("utf-8")).hexdigest()

    response = requests.get(
        BASE_URL,
        params=params,
        auth=AUTH,
    )
    response.raise_for_status()
    
    event = pd.read_csv(StringIO(response.text))
    return event


In [217]:
# def add_median_times_to_studies():
for i in range(200, len(studies)):

    study = studies.iloc[i]

    if not (pd.isna(study["median_gps_interval_seconds"]) or study["median_gps_interval_seconds"] == -1):
        print(f"{i}/{len(studies)} study {_id}, {study['name']}, {study['number_of_individuals']} individuals, {study['median_gps_interval_seconds']}")
        continue
    
    _id = study["id"]
    print(f"{i}/{len(studies)} fetching study {_id}, {study['name']}, {study['number_of_individuals']} individuals")

    try:

        event = get_events(study["id"])
    
        print(f"fetched {len(event)} rows, {len(event['tag_local_identifier'].unique())} unique tags")
    
        event["timestamp"] = pd.to_datetime(event["timestamp"])
        median_gps_interval_seconds = event.sort_values(by = ['tag_local_identifier', 'timestamp'])["timestamp"].diff().median().total_seconds()
    
        studies.at[i, "median_gps_interval_seconds"] = median_gps_interval_seconds
    
        print(f"updated {i}/{len(studies)} to median {median_gps_interval_seconds} seconds")
    except Exception as e:
        print(f"error {e}")
        studies.loc[i, "error"] = str(e)
        studies.loc[i, "median_gps_interval_seconds"] = -1

    studies.to_csv(FOLDER + "studies_with_gps_interval.csv")

200/320 study 466196195, Latham Alberta Wolves, 12.0 individuals, 7215.0
201/320 study 466196195, African Lion - Panthera Lion Central Kalahari Game Reserve, 13.0 individuals, 1800.0
202/320 study 466196195, Migrations of Caspian terns in family groups, 27.0 individuals, 600.0
203/320 study 466196195, Eurasian wigeon (Mareca penelope) Netherlands Lithuania 2018-2019, 39.0 individuals, 1.0
204/320 study 466196195, Wood stork (Mycteria americana) Southeastern US 2004-2019, 133.0 individuals, 3600.0
205/320 study 466196195, GPS tracking of bobcats and coyotes in northern Washington, 70.0 individuals, 14400.0
206/320 study 466196195, Variability of White Stork flight patterns prior to earthquakes, 15.0 individuals, 300.0
207/320 study 466196195, Long-billed Curlew Migration from the Intermountain West, 100.0 individuals, 3086.0
208/320 study 466196195, Elk in southwestern Alberta, 171.0 individuals, 7200.0
209/320 study 466196195, Brown pelicans in the Gulf of Mexico (data from Geary et al

In [5]:
studies_loaded = pd.read_csv(FOLDER + "studies_with_gps_intervals.csv",
                             parse_dates=['timestamp_last_deployed_location', 'timestamp_first_deployed_location'])

In [6]:
studies = studies_loaded

In [12]:
keepers = studies[
    (studies["median_gps_interval_seconds"] <= 3600) &
    (studies["median_gps_interval_seconds"] > 0) &
    (studies["license_type"] != "CUSTOM") &
    (studies["number_of_individuals"] >= 5) &
    (studies["number_of_individuals"] >= 5) &
    (studies["id"] != 468460067) & # two versions of this study exist, skip this one.
    ~pd.isna(studies["taxon_ids"]) &
    True
].copy().reset_index()
keepers.to_csv(FOLDER + "studies_to_use.csv")
